# Loop0, stage 00: read the wells, label them, concatenate

The first and largest loop.  It pools every plate acquired up to that point, draws the gating strategy by hand, builds the fluorescence-minus-one calibration that every later loop reuses, and drops the experiments that were not part of the screen.

## Inputs

| file | produced by | holds |
|---|---|---|
| `experiment_table.csv` | this folder | maps an acquisition date and time to an experiment number |
| `metadata.csv` | this folder | culture conditions, one row per experiment |
| `$LABCOMPASS_FCS_ROOT/` | acquisition, outside the repository | one unmixed `.fcs` file per well |

## Outputs

| file | holds |
|---|---|
| `Loop0.h5ad` | every cell of every well, labelled, with the antibody channels only |
| `Loop0_500k.h5ad` | about 500,000 of those cells, spread evenly over the experiments |

Run the cells from top to bottom.  Nothing below reads a file that a later cell
writes, so `Restart Kernel and Run All Cells` reproduces both outputs.

## Setup

Every path and every number that this notebook uses is set here.

In [ ]:
import gc
import os
import csv

import pytometry as pm

from labcompass import (
    add_live_dead_obs,
    attach_condition_metadata,
    attach_cytometer_metadata,
    concat_wells,
    downsample,
    drop_channels,
    find_fcs_files,
    load_metadata_table,
    read_well,
    rename_channels_to_antibody,
    target_size_for,
)

# ---------------------------------------------------------------- configuration
# Where the unmixed .fcs files live.  They are far too large to commit, so the
# repository only records the path.  Override it without editing this notebook
# by setting the environment variable LABCOMPASS_FCS_ROOT.
FCS_ROOT = os.environ.get("LABCOMPASS_FCS_ROOT", '/rds/project/rds-SDzz0CATGms/unsorted/for_Juan_Licyel')
DATA_PATH = os.path.join(FCS_ROOT, '')
METADATA_CSV = 'metadata.csv'

OUT_ALL = "Loop0.h5ad"
OUT_SUBSET = "Loop0_500k.h5ad"

# Cap the downsampled file at this many cells in total, spread evenly over the
# experiments.  It exists so the gating notebook has something small to plot.
DESIRED_CELL_NUMBER = 500000
DOWNSAMPLE_SEED = 0

## The wells to read

Every unmixed `.fcs` file below `DATA_PATH` is read.
Only files inside a directory called `filtered` are taken, because that is where the unmixing software writes its output.
The list is sorted, so the order does not depend on the filesystem.

In [ ]:
files = find_fcs_files(DATA_PATH, subdir='filtered')
print(f"{len(files)} wells to read")
if not files:
    raise FileNotFoundError(f"no .fcs files found under {DATA_PATH}")

## The condition table

One row per experiment, holding the culture conditions.  The index is read as
text, so an experiment number matches whether it is written as `239` or `"239"`.

In [ ]:
metadata = load_metadata_table(METADATA_CSV)
print(f"{len(metadata)} experiments, {len(metadata.columns)} condition columns")
metadata.head()

In [ ]:
# One row per acquisition, holding the date and the start time that identify the
# .fcs file, together with the experiment number and the bead and cell counts.
with open("experiment_table.csv", newline="") as handle:
    experiment_list = list(csv.DictReader(handle))

print(f"{len(experiment_list)} acquisitions, "
      f"{len({row['experiment_number'] for row in experiment_list})} experiments")

## Read each well and label it

Each well is matched to an experiment by its acquisition date and start time,
which the cytometer wrote into the `.fcs` file.  A matched row is removed from
`experiment_list` so that two wells cannot claim the same acquisition.

A well whose acquisition is not in the table, or whose experiment number is not
in the condition table, is skipped.  The acquisitions left over at the end are
printed, so an unmatched well is visible rather than silent.

`pytometry.pp.split_signal` moves the scatter and time channels from `.X` into
`.obs` and keeps only the area measurements in `.X`.

In [ ]:
adatas = []
skipped = []

for path in files:
    adata = read_well(path)
    if adata.n_obs == 0:
        continue

    target_date = adata.uns["meta"]["date"]
    target_btim = adata.uns["meta"]["btim"]

    match = next(
        (i for i, row in enumerate(experiment_list)
         if row["date"] == target_date and row["btim"] == target_btim),
        None,
    )
    if match is None:
        skipped.append((path, "acquisition not in experiment_table.csv"))
        continue

    row = experiment_list[match]
    if row["experiment_number"] not in metadata.index:
        skipped.append((path, f"experiment {row['experiment_number']} not in {METADATA_CSV}"))
        continue

    adata.obs["experiment_number"] = int(row["experiment_number"])
    adata.obs["replicate"] = int(row["replicate"])
    adata.obs["count_beads"] = row["count_beads"]
    adata.obs["cell_counts"] = row["cell_counts"]
    attach_cytometer_metadata(adata)
    attach_condition_metadata(adata, metadata, row["experiment_number"])

    pm.pp.split_signal(adata, var_key="channel", option="area", data_type="facs")
    adatas.append(adata)

    # Remove the matched acquisition so a second well cannot claim it.
    del experiment_list[match]

del adata
gc.collect()

print(f"{len(adatas)} wells kept, {len(skipped)} skipped")
for path, reason in skipped:
    print("  skipped", path, "|", reason)
print(f"{len(experiment_list)} acquisitions in the table were never matched to a well")

## Concatenate, move the live/dead channel, drop the channels we do not analyse

`concat_wells` keeps only the channels present in every well and makes the cell
names unique across wells.  The live/dead channel is copied into `.obs["7-AAD"]`
before it is dropped, because the live gate is drawn against it.

In [ ]:
adata_all = concat_wells(adatas)
del adatas
gc.collect()
adata_all

In [ ]:
add_live_dead_obs(adata_all)
adata_all = drop_channels(adata_all)
fluorochromes = rename_channels_to_antibody(adata_all)

print("antibodies:  ", list(adata_all.var_names))
print("fluorochromes:", fluorochromes)
adata_all

## Write the full object

This is the input of stage 02.

In [ ]:
adata_all.write(OUT_ALL)
print(f"wrote {OUT_ALL}: {adata_all.n_obs:,} cells x {adata_all.n_vars} antibodies")

## Write the downsampled object

Each experiment is capped at the same number of cells.  An experiment with fewer
cells than the cap is kept whole, so the total comes out below the target.

In [ ]:
target_size = target_size_for(adata_all, DESIRED_CELL_NUMBER, obs="experiment_number")
print(f"cap of {target_size:,} cells per experiment")

adata_small = downsample(
    adata_all, obs="experiment_number", target_size=target_size, seed=DOWNSAMPLE_SEED
)
adata_small.write(OUT_SUBSET)
print(f"wrote {OUT_SUBSET}: {adata_small.n_obs:,} cells")